# models

> Lean data shapes for the transcript-decomposition pipeline: in-core mirrors of the
> forced-alignment / VAD / text DTOs (no FastHTML deps), run configuration, the committed
> graph-segment carrier, and the decomposition run manifest (proto-bundle).

In [ ]:
#| default_exp models

In [ ]:
#| export
import json
import time
import uuid
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

In [ ]:
#| export
@dataclass
class FAWord:
    """One word-level forced-alignment result (segment-local times).

    Lean in-core mirror of the forced-alignment capability's `ForcedAlignItem`
    wire shape, so the core needs no interface-library dependency for a
    three-field DTO (pass-2 evidence: the typed FA result belongs to the
    forced-alignment adapter interface).
    """
    text: str          # The aligned word (punctuation typically stripped by the model)
    start_time: float  # Word start in seconds (relative to the segment audio)
    end_time: float    # Word end in seconds (relative to the segment audio)

    @classmethod
    def from_wire(
        cls,
        d: Dict[str, Any],  # Capability wire dict
    ) -> "FAWord":  # Parsed word (extra keys ignored)
        """Build from a capability wire dict, tolerating extra keys."""
        return cls(
            text=str(d.get("text", "")),
            start_time=float(d.get("start_time", 0.0)),
            end_time=float(d.get("end_time", 0.0)),
        )

In [ ]:
#| export
@dataclass
class VADChunk:
    """A voice-activity time range within one pipeline segment (segment-local)."""
    index: int         # Chunk index within the pipeline segment
    start_time: float  # Start in seconds (relative to the segment audio)
    end_time: float    # End in seconds (relative to the segment audio)

    @property
    def duration(self) -> float:  # Chunk duration in seconds
        """Chunk duration."""
        return self.end_time - self.start_time

In [ ]:
#| export
@dataclass
class TextSegment:
    """A text segment produced by alignment, before graph commit.

    Lean in-core mirror of the page-centric `TextSegment` (which imports FastHTML
    card-stack types); the core carries only the alignment-relevant fields
    (E11 precedent — keep core logic free of UI-adjacent deps).
    """
    index: int                                # Position within its pipeline segment
    text: str                                 # Segment text content
    source_id: Optional[str] = None           # Upstream transcription row id (job_id)
    source_provider_id: Optional[str] = None  # Producing capability name
    start_char: Optional[int] = None          # Start char offset into the pipeline-segment text
    end_char: Optional[int] = None            # End char offset into the pipeline-segment text

In [ ]:
#| export
@dataclass
class SegmentVariant:
    """One transcriber's text + char range for one fine segment (stage 5).

    Becomes a `GraphNodeRef(Transcript) + CharSlice` provenance ref on the
    committed Segment node — text is stored ONCE per transcriber at the coarse
    Transcript node; fine-grained variants are SLICES, never duplicated text
    (Thread-1 slices-until-promoted)."""
    transcriber: str            # Transcriber capability name
    text: str                   # This transcriber's text for the chunk
    start_char: Optional[int]   # Char offset into that transcriber's pipeline-segment text
    end_char: Optional[int]     # End char offset

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form
        """Serialize to a plain dict."""
        return asdict(self)


@dataclass
class DecompSegment:
    """One fine spine segment (stage 5: shared audio-side skeleton + per-transcriber variants).

    Identity is AUDIO-SIDE (owning pipeline segment + chunk range) — shared
    across transcribers by construction; `text` is the AUTHORITATIVE
    (accuracy-transcriber) alignment, with every transcriber's chunk text
    riding `variants` (the authoritative one included)."""
    index: int             # Source-wide 0-based fine-spine position
    text: str              # Authoritative text (the --text-from transcriber's alignment; "" = no aligned words)
    start_time: float      # Start in source-audio seconds (navigation)
    end_time: float        # End in source-audio seconds
    chunk_start: float     # VAD chunk start (chunk-local seconds within the pipeline-segment WAV)
    chunk_end: float       # VAD chunk end (chunk-local seconds)
    vad_chunk_index: int   # Chunk index within its pipeline segment
    pseg_index: int        # Owning pipeline-segment (AudioSegment) index within the source
    variants: List[SegmentVariant] = field(default_factory=list)  # Per-transcriber chunk texts + char ranges

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form
        """Serialize to a plain dict."""
        return asdict(self)

In [ ]:
#| export
@dataclass
class DecompConfig:
    """Configuration for one transcript-decomposition run."""
    vad_capability: str = "cjm-capability-silero-vad"                    # VAD capability id
    fa_capability: str = "cjm-capability-qwen3-forced-aligner"  # Forced-alignment capability id
    graph_capability: str = "cjm-capability-graph-sqlite"                     # Graph-storage capability id
    text_from: Optional[str] = None  # Authoritative transcriber for layer-0 text (None: sole transcriber; REQUIRED for multi-transcriber manifests)
    language: str = "English"  # Forced-alignment language
    media_type: str = "audio"  # Source media type
    force: bool = False        # Bypass capability-side caches (VAD + FA)
    assume_yes: bool = False   # Auto-accept HITL seams (headless mode)

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict snapshot for the manifest
        """Serialize to a plain dict."""
        return asdict(self)

In [ ]:
#| export
@dataclass
class DecompSourceRecord:
    """Record of one Source whose fine spine this run committed (stage 5:
    `Document` dissolved into `Source` — decomp EXTENDS the transcription-
    emitted root rather than creating a document)."""
    source_node_id: str  # Graph Source node id (deterministic; recomputable from the content hash)
    source_path: str     # Originating source audio path (from the upstream manifest)
    title: str           # Source display title
    segment_count: int   # Number of fine Segment nodes committed
    segment_ids: List[str] = field(default_factory=list)  # Graph Segment node ids, in order

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form
        """Serialize to a plain dict."""
        return asdict(self)

In [ ]:
#| export
@dataclass
class DecompManifest:
    """Durable record of one decomposition run (proto-bundle; see CR-20).

    Schema 0.2.0 (stage 5): `documents` became `sources` (Document dissolved
    into Source); entries carry the deterministic Source node id."""
    run_id: str             # Unique run identifier
    created_at: float       # Unix timestamp at run start
    config: Dict[str, Any]  # DecompConfig snapshot
    source_manifest: str    # Path to the consumed transcription run manifest
    source_format: str = ""  # Upstream manifest format tag (interchange contract)
    source_version: str = ""  # Upstream manifest schema version
    capabilities: Dict[str, Dict[str, Any]] = field(default_factory=dict)  # instance_id -> identity/db_path/config_hash
    sources: List[DecompSourceRecord] = field(default_factory=list)   # Extended sources, input order

    FORMAT: str = field(default="cjm-transcript-decomp-core/run-manifest", repr=False)  # Format tag
    VERSION: str = field(default="0.2.1", repr=False)                                   # Schema version

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form for JSON serialization
        """Serialize to a plain dict with nested sources."""
        return {
            "format": self.FORMAT,
            "version": self.VERSION,
            "run_id": self.run_id,
            "created_at": self.created_at,
            "config": self.config,
            "source_manifest": self.source_manifest,
            "source_format": self.source_format,
            "source_version": self.source_version,
            "capabilities": self.capabilities,
            "sources": [s.to_dict() for s in self.sources],
        }

    def save(
        self,
        path: Union[str, Path],  # Destination JSON file (parent dirs created)
    ) -> Path:  # The written path
        """Write the manifest as pretty-printed JSON."""
        out = Path(path)
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(json.dumps(self.to_dict(), indent=2))
        return out

In [ ]:
#| export
def new_run_id() -> str:  # e.g. "decomp_20260607_153000_1a2b3c4d"
    """Generate a unique, sortable decomposition run id."""
    return f"decomp_{time.strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

In [ ]:
# models smoke checks
w = FAWord.from_wire({"text": "Hello", "start_time": 0.1, "end_time": 0.4, "extra": 1})
assert w.text == "Hello" and w.end_time == 0.4
c = VADChunk(index=0, start_time=1.0, end_time=2.5)
assert abs(c.duration - 1.5) < 1e-9

# stage-5 segment shape: audio-side identity fields + per-transcriber variants
seg = DecompSegment(index=0, text="hi there", start_time=10.0, end_time=12.0,
                    chunk_start=1.0, chunk_end=3.0, vad_chunk_index=0, pseg_index=0,
                    variants=[SegmentVariant("whisper", "hi there", 0, 8),
                              SegmentVariant("voxtral", "Hi, there", 0, 9)])
d = seg.to_dict()
assert d["chunk_start"] == 1.0 and d["pseg_index"] == 0
assert d["variants"][1]["transcriber"] == "voxtral"

assert new_run_id().startswith("decomp_")
m = DecompManifest(run_id="r", created_at=0.0, config={}, source_manifest="/tmp/s.json")
md = m.to_dict()
assert md["format"] == "cjm-transcript-decomp-core/run-manifest"
assert md["version"] == "0.2.1" and md["sources"] == []
rec = DecompSourceRecord(source_node_id="sid", source_path="/a.mp3", title="a",
                         segment_count=2, segment_ids=["s1", "s2"])
assert rec.to_dict()["source_node_id"] == "sid"
print("models checks OK")